# Judge a pool with llmjudge

Runtime → Change runtime type → **A100** or **L4**, then run the cells top to bottom.

**Code comes from GitHub. Drive holds only data** — the rows going in, the
results coming out. Nothing else is uploaded, and the runtime can die without losing
anything.

Two Colab Secrets (the key icon in the sidebar, then toggle notebook access):

| secret | what |
|---|---|
| `GH_TOKEN` | fine-grained PAT, **Contents: Read-only**, scoped to this one repo |
| `HF_TOKEN` | for the MedGemma weights |

Neither is ever typed into a cell, so a shared notebook carries no credential.

Before the first run, put the rows on Drive. A CSV is enough — upload it to
`MyDrive/judge/` at drive.google.com and the judge reads every row, every column a
field:

```
question,candidate_answer
What is 2 + 2?,4
```

For a stratified, weighted sample of a large table, draw the pool first and upload that
`.jsonl` instead (114 KB for the pilot):

```bash
llmjudge make-items --spec configs/pool.diabetes130.toml --pool pilot \
    --root /path/to/your/runs --out items-pilot.jsonl
```

## 1. Install, and mount Drive

In [ ]:
REPO  = 'your-org/llmjudge'                    # <- set once
TAG   = 'v0.1.0'
DRIVE = '/content/drive/MyDrive/judge'

from google.colab import drive, userdata
drive.mount('/content/drive')

!pip -q install "git+https://{userdata.get('GH_TOKEN')}@github.com/{REPO}.git@{TAG}"

import llmjudge; print('llmjudge', llmjudge.__version__ if hasattr(llmjudge, '__version__') else '', llmjudge.__file__)

## 2. The pool

Straight off Drive, a `.csv` or a `.jsonl`. The judge reads it from there; only the
results are written to `/content` and mirrored back.

In [ ]:
ITEMS = f'{DRIVE}/rows.csv'                     # or items-pilot.jsonl from make-items

import collections
from llmjudge.run import iter_objs             # reads either format, the judge's own parser

rows = [obj for _, obj in iter_objs(ITEMS)]
print(len(rows), 'rows  ', dict(collections.Counter(r.get('group', 'all') for r in rows)))
print(len(rows[0]['fields']), 'columns')

## 3. Serve MedGemma

~10 minutes the first time: the weights download, then vLLM loads them. The script
ships inside the package. It exports `LLMJUDGE_BASE_URL`, `LLMJUDGE_API_KEY` and
`LLMJUDGE_MODEL`, so the judge cell says nothing about the server.

Judging against a vendor API instead? Skip this cell and set those three environment
variables yourself.

In [ ]:
import os, llmjudge
SERVE = os.path.join(os.path.dirname(llmjudge.__file__), 'serve_vllm.py')
%run {SERVE}

## 4. Judge

Results are written to `/content`, where append and fsync mean what they say, and the
tail is copied up to Drive every 100 seconds and once more at the end. Re-running this
cell after a disconnect resumes: rows already on Drive are not re-sent.

**Two runtimes at once?** Set `PART` to `'1/2'` in one and `'2/2'` in the other, and give
each a different `OUT`. The send order is shuffled before it is cut, so each gets a
random half and no row is judged twice; `cat` the two `results.jsonl` files together
afterwards. One `OUT` per runtime, always — two of them appending to one file tear
each other's lines.

Exit code `0` is every planned row recorded, `3` stopped incomplete, `4` judged but not
all on Drive — `4` is the one to care about, and it names where the rows still are.

In [ ]:
from llmjudge.colab import run

OUT  = f'{DRIVE}/results/pilot'
PART = None                     # '1/2' here and '2/2' in a second runtime, each with its own OUT

exit_code = run(items=ITEMS,
                out=OUT,
                run_tag='pilot-01',
                prompt='c3-reasoning-2',
                guided=False,
                max_tokens=2560,
                part=PART)
print('exit', exit_code)

## 5. What it said

In [ ]:
import json

with open(f'{OUT}/summary.json') as f:
    s = json.load(f)

print(s['rows'], 'rows,', s['missing'], 'missing,',
      s['parse_errors'], 'parse errors,', s['errors'] or 'no errors',
      f"(part {s['part']})" if s.get('part') else '')

for group, g in sorted(s['by_group'].items()):
    print(f"\n{group}  ({g['judged']}/{g['rows']} judged)")
    for label, rate in sorted(g['rates'].items()):
        print(f"   {label:<14} {rate:7.1%}   weighted {g['rates_weighted'][label]:7.1%}")

---

**Sizing a real run.** The last full run did 5,500 rows in 1h50m on one A100 — about
50 rows/minute, p50 latency 20 s. The `full` pool is 55,039 rows, so ~18 hours on one
runtime. Two ways to cut that: `PART` across two runtimes at once (`'1/2'` and `'2/2'`,
one `OUT` each), or several sessions in a row resuming into the same Drive folder —
a re-run never re-sends a row that is already answered.